In [ ]:
import os
import yaml
import numpy as np
import polars as pl
import pandas as pd
from tqdm import tqdm
import statsmodels.api as sm
from huggingface_hub import hf_hub_download
import pyranges as pr

from plotnine import *
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype'] = 'none'

# Get variant lists from all benchmarks

## UKBBGym

In [ ]:
# Get gene trait associations
LOCAL_DATA_DIR = "PATH_TO_FILE"
ASSOC_PATH = f'{LOCAL_DATA_DIR}/regenie_127phenotypes_lofteeHC_mac20_EUR_miss20per.parquet'
CORR_PATH  = f'{LOCAL_DATA_DIR}/regenie_127phenotypes_mac20_lofteeHC_EUR_correlations.parquet'

gene_trait_df = (
    pl.read_parquet(ASSOC_PATH)
    .filter(pl.col('pval_fdr') <= 0.05)
    .select(['region', 'phenotype', 'pval_fdr'])
)

loftee_corrs = (
    pl.read_parquet(CORR_PATH)
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir'])
)

gene_trait_df = (
    gene_trait_df
    .join(loftee_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
    .sort('loftee_corr_abs', descending=True)
    .unique(subset=["region"], keep="first", maintain_order=True)
)
gene_trait_df

In [ ]:
LOCAL_DATA_DIR = "PATH_TO_FILE"
ASSOC_PATH = f'{LOCAL_DATA_DIR}/proteomics_prs_df_loftee_mac20_burden_regression_results.parquet'
CORR_PATH  = f'{LOCAL_DATA_DIR}/olink_all_mac20_lofteeHC_correlations.parquet'

olink_whitelist = (
    pl.read_parquet(f'{ASSOC_PATH}')
    .rename({'gene': 'region'})
    .filter((pl.col('padj')<=0.05) & (pl.col('wilcox_padj')<=0.05) )
    .select(['region'])
    .with_columns(
        phenotype = pl.col('region') + '_olink'
    )
)

olink_corrs = (
    pl.read_parquet(f'{CORR_PATH}')
    .filter(pl.col('correlation')>0)
    .with_columns(
        loftee_corr = pl.col('correlation'),
        loftee_corr_abs = pl.col('correlation').abs(),
        loftee_corr_dir = pl.col('correlation')/pl.col('correlation').abs(),
    )
    .drop_nans()
    # .select(['region', 'phenotype', 'loftee_corr', 'loftee_corr_abs', 'loftee_corr_dir']) 
)

olink_whitelist = (
    olink_whitelist
    .join(olink_corrs, on=['region', 'phenotype'], how='inner')
    .drop_nans()
    .sort('loftee_corr_abs', descending=True)
    .unique(subset=["region"], keep="first", maintain_order=True)
)

olink_whitelist.sort('n_variants')

In [ ]:
required_genes_df = pl.concat([gene_trait_df[['region']], olink_whitelist[['region']]]).unique()

required_genes_df

In [ ]:
LOCAL_ANNO_DIR = "PATH_TO_FILE"
ANNO_FILE = "annotations_no_dup_20260630.parquet"

mac = 20

anno = (
    pl.scan_parquet(f"{LOCAL_ANNO_DIR}/{ANNO_FILE}")
    .with_columns(
        id = pl.col('id') + '_' + pl.col('region'),
        proximal_promoter = pl.col('dist_to_tss').abs() <= 500,
        encode_promoter = pl.col('encode_pls'),
        encode_enhancer = pl.col('encode_pels') | pl.col('encode_dels'),
    )
    .with_columns(
        # promoter_variant = pl.col('proximal_promoter') | pl.col('encode_promoter'),
        promoter_variant = pl.col('proximal_promoter'),
        enhancer_variant = pl.col('encode_enhancer')
    )
    .join(
        required_genes_df.lazy(),
        on='region',
        how='semi'
    )
    .filter(
        pl.col('ac_ukb') <= mac
    )
)

consequence_cols = [c for c in anno.collect_schema().names() if c.startswith('consequence_')]
ccre_cols = ['promoter_variant', 'enhancer_variant']

ukbgym = (
    anno
    .select(
        ['id'] + consequence_cols + ccre_cols
    )

    .unpivot(
        index = ['id'],
        variable_name="consequence",
        value_name="is_consequence"
    )
    .filter(
        pl.col('is_consequence') == 1
    )
    .with_columns(
        consequence = pl.col('consequence').str.replace('consequence_', ''),
        benchmark = pl.lit('UKBBGym')
    )
    .drop(['is_consequence'])
    .unique()
    .collect()
)

# anno.write_parquet('PATH_TO_FILE')
ukbgym

In [ ]:
ukbgym['consequence'].value_counts(sort=True)

## ClinVar

In [ ]:
# CLINVAR_PATH = 'PATH_TO_FILE'
CLINVAR_PATH = 'PATH_TO_FILE'

clinvar = (
    pl.scan_parquet(f"{CLINVAR_PATH}")
    .with_columns(
        id = pl.col('id') + '_' + pl.col('region'),
        proximal_promoter = pl.col('dist_to_tss').abs() <= 500,
        encode_promoter = pl.col('encode_pls'),
        encode_enhancer = pl.col('encode_pels') | pl.col('encode_dels'),
    )
    .with_columns(
        # promoter_variant = pl.col('proximal_promoter') | pl.col('encode_promoter'),
        promoter_variant = pl.col('proximal_promoter'),
        enhancer_variant = pl.col('encode_enhancer')
    )
)

consequence_cols = [c for c in clinvar.collect_schema().names() if c.startswith('consequence_')]
ccre_cols = ['promoter_variant', 'enhancer_variant']

clinvar = (
    clinvar
    .select(
        ['id'] + consequence_cols + ccre_cols
    )

    .unpivot(
        index = ['id'],
        variable_name="consequence",
        value_name="is_consequence"
    )
    .filter(
        pl.col('is_consequence') == 1
    )
    .with_columns(
        consequence = pl.col('consequence').str.replace('consequence_', ''),
        benchmark = pl.lit('ClinVar\n(non VUS)')
    )
    .drop(['is_consequence'])
    .unique()
    .collect()
)

clinvar

## ProteinGym

In [ ]:
PG_DIR = 'PATH_TO_FILE'
dms_dir = f'{PG_DIR}/DMS_ProteinGym_substitutions/'

pg_list = []
for filename in tqdm(os.listdir(dms_dir)):
    if 'HUMAN' in filename:
        if filename.endswith('.csv'):
            temp = pl.read_csv(f'{dms_dir}{filename}')
            big_name = filename.split('.')[0]
            temp = temp.with_columns(
                pl.lit(big_name.split('_')[0]).alias('protein_name'),
                # pl.lit('_'.join(big_name.split('_')[2:])).alias('experiment_name'),
                pl.lit(filename.split('.')[0]).alias('file_name'),
                )
            pg_list.append(temp)

pgdf = pl.concat(pg_list)

print(f"Number of unique proteins in ProteinGym: {pgdf['protein_name'].n_unique()}")

protein_gym_snv = (
    pgdf
    .with_columns(
        id = pl.col('mutant') + '_' + pl.col('protein_name')
    )
    .select(['id'])
    .unique()
)

protein_gym_snv

In [ ]:
dms_indel_dir = f'{PG_DIR}/DMS_ProteinGym_indels/'

pg_list = []
for filename in tqdm(os.listdir(dms_indel_dir)):
    if 'HUMAN' in filename:
        if filename.endswith('.csv'):
            temp = pl.read_csv(f'{dms_indel_dir}{filename}')
            big_name = filename.split('.')[0]
            temp = temp.with_columns(
                pl.lit(big_name.split('_')[0]).alias('protein_name'),
                # pl.lit('_'.join(big_name.split('_')[2:])).alias('experiment_name'),
                pl.lit(filename.split('.')[0]).alias('file_name'),
                )
            pg_list.append(temp)

pg_indel = (
    pl.concat(pg_list)
    .with_columns(
        mutant = pl.col('mutated_sequence')
                      .rank(method='max', descending=False)
                      .over(['protein_name', 'file_name'])
                      .cast(pl.Utf8)
    )
)

print(f"Number of unique proteins in ProteinGym: {pg_indel['protein_name'].n_unique()}")

protein_gym_indel = (
    pg_indel
    .with_columns(
        id = pl.col('mutant') + '_' + pl.col('protein_name')
    )
    .select(['id'])
    .unique()
)

protein_gym_indel

In [ ]:
protein_gym = (
    pl.concat([protein_gym_snv, protein_gym_indel]).unique()
    .with_columns(
        consequence = pl.lit('missense_variant'),
        benchmark = pl.lit('ProteinGym\n(Human)')
    )
)
protein_gym

## TraitGym

In [ ]:
# Download the file to your local cache and get the path
file_path = hf_hub_download(
    repo_id="songlab/TraitGym", 
    filename="mendelian_traits_matched_9/test.parquet",
    repo_type="dataset"
)

# Read the local file
trait_gym_m = (
    pl.read_parquet(file_path)
    .with_columns(
        id = pl.concat_str([
            pl.col("chrom").cast(pl.Utf8),
            pl.lit(":"),
            pl.col("pos").cast(pl.Utf8),
            pl.lit(":"),
            pl.col("ref"),
            pl.lit(":"),
            pl.col("alt")
        ]),
        consequence = pl.col("consequence").str.to_lowercase(),
        is_consequence = pl.lit(1)
    )
    .select(['id', 'consequence', 'is_consequence'])
    .pivot(
        index = 'id',
        on = 'consequence',
        values = 'is_consequence',
    )
    # After the pivot, absent consequences are null (not 0), so fill_null before OR-ing.
    .with_columns(
        promoter_variant = pl.max_horizontal('pls', 'pls_flank').fill_null(0),
        enhancer_variant = pl.max_horizontal('pels', 'pels_flank', 'dels', 'dels_flank').fill_null(0),
    )
    .drop(['pls', 'pls_flank', 'pels', 'pels_flank', 'dels', 'dels_flank'])
    .unpivot(
        index = ['id'],
        variable_name="consequence",
        value_name="is_consequence"
    )
    .filter(
        pl.col('is_consequence') == 1
    )
    .with_columns(
        consequence = pl.col('consequence').str.replace('consequence_', ''),
        benchmark = pl.lit('TraitGym\nMendelian')
    )
    .drop(['is_consequence'])
    .unique()
)

trait_gym_m

In [ ]:
file_path = hf_hub_download(
    repo_id="songlab/TraitGym", 
    filename="complex_traits_matched_9/test.parquet",
    repo_type="dataset"
)

pl.read_parquet(file_path)['consequence'].unique().to_list()

In [ ]:
# Download the file to your local cache and get the path
file_path = hf_hub_download(
    repo_id="songlab/TraitGym", 
    filename="complex_traits_matched_9/test.parquet",
    repo_type="dataset"
)

# Read the local file
trait_gym_c = (
    pl.read_parquet(file_path)
    .with_columns(
        id = pl.concat_str([
            pl.col("chrom").cast(pl.Utf8),
            pl.lit(":"),
            pl.col("pos").cast(pl.Utf8),
            pl.lit(":"),
            pl.col("ref"),
            pl.lit(":"),
            pl.col("alt")
        ]),
        consequence = pl.col("consequence").str.to_lowercase(),
        is_consequence = pl.lit(1)
    )
    .select(['id', 'consequence', 'is_consequence'])
    .pivot(
        index = 'id',
        on = 'consequence',
        values = 'is_consequence',
    )
    # After the pivot, absent consequences are null (not 0), so fill_null before OR-ing.
    .with_columns(
        promoter_variant = pl.max_horizontal('pls', 'pls_flank', 'dnase-h3k4me3', 'dnase-h3k4me3_flank').fill_null(0),
        enhancer_variant = pl.max_horizontal('pels', 'pels_flank', 'dels', 'dels_flank').fill_null(0),
    )
    .drop(['pls', 'pls_flank', 'dnase-h3k4me3', 'dnase-h3k4me3_flank', 'pels', 'pels_flank', 'dels', 'dels_flank'])
    .unpivot(
        index = ['id'],
        variable_name="consequence",
        value_name="is_consequence"
    )
    .filter(
        pl.col('is_consequence') == 1
    )
    .with_columns(
        consequence = pl.col('consequence').str.replace('consequence_', ''),
        benchmark = pl.lit('TraitGym\nComplex')
    )
    .drop(['is_consequence'])
    .unique()
)

trait_gym_c

# Plotting

In [ ]:
consequence_map = {
    # === CDS (Coding Sequence) ===
    # UKBGym/ClinVar/ProteinGym VEP columns
    'missense_variant': 'Coding',
    'frameshift_variant': 'Coding',
    'synonymous_variant': 'Coding',
    'stop_gained': 'Coding',
    'stop_lost': 'Coding',
    'start_lost': 'Coding',
    'start_retained_variant': 'Coding',
    'stop_retained_variant': 'Coding',
    'incomplete_terminal_codon_variant': 'Coding',
    'coding_sequence_variant': 'Coding',
    'protein_altering_variant': 'Coding',
    'inframe_deletion': 'Coding',
    'inframe_insertion': 'Coding',
    'transcript_ablation': 'Coding',
    

    # === Considering all Splicing as Coding for this plot ===
    # UKBGym VEP columns
    'splice_donor_variant': 'Coding',
    'splice_acceptor_variant': 'Coding',
    'splice_region_variant': 'Coding',
    'splice_donor_5th_base_variant': 'Coding',
    'splice_donor_region_variant': 'Coding',
    'splice_polypyrimidine_tract_variant': 'Coding',

    # === UTR (Untranslated Regions) ===
    # UKBGym VEP columns
    '3_prime_utr_variant': 'UTR',
    '5_prime_utr_variant': 'UTR',

    # === Promoter ===
    # UKBGym, ClinVar: defined as within 2kb upstream of TSS (using dist_to_tss)
    'promoter_variant': 'Promoter',
    
    'enhancer_variant': 'Enhancer',

    # === Introns (non-coding gene body) ===
    # UKBGym VEP columns
    'intron_variant': 'Introns',
    'non_coding_transcript_exon_variant': 'Introns',    # Exons of non-coding transcripts (gene body)
    'non-coding_transcript_variant': 'Introns',         # Non-coding transcript (gene body)
    'genic_upstream_transcript_variant': 'Introns',     # Within gene body, upstream of a transcript
    'genic_downstream_transcript_variant': 'Introns',   # Within gene body, downstream of a transcript

    # === Intergenic (outside gene body) ===
    # UKBGym VEP columns
    'upstream_gene_variant': 'Intergenic',  # 5kb upstream — too broad for Promoter
    'downstream_gene_variant': 'Intergenic',
    'intergenic_variant': 'Intergenic',
    'regulatory_region_variant': 'Intergenic',
    'no_sequence_alteration': 'Intergenic',
    'ctcf-only': 'Intergenic',
    'ctcf-only_flank': 'Intergenic'
}

# Across benchmark plot

In [ ]:
all_benchmarks = pl.concat([ukbgym, clinvar, protein_gym, trait_gym_m, trait_gym_c])
all_benchmarks

In [ ]:
num_df = (
    all_benchmarks
    .group_by(['benchmark', 'consequence'])
    .agg(
        num_variants = pl.len()
    )
    .with_columns(
        consequence_category = pl.col('consequence').replace_strict(consequence_map, default="Other")
    )
)

num_df

In [ ]:
num_df['consequence_category'].value_counts(sort=True)

In [ ]:
plot_df = (
    num_df
    .group_by(['benchmark', 'consequence_category'])
    .agg(
        num_variants = pl.sum('num_variants')
    )
)

plot_df

# Across benchmark plots

In [ ]:
from matplotlib import colormaps
from matplotlib.colors import to_hex

CMAP = "magma"
cmap = colormaps[CMAP]

positions = {
        "UKBBGym":             0.30,
        "ClinVar\n(non VUS)":  0.55,
        "ProteinGym\n(Human)": 0.75,
        "TraitGym\nMendelian": 0.85,
        "TraitGym\nComplex":   0.92,
}
benchmark_colors = {k: to_hex(cmap(v)) for k, v in positions.items()}

all_categories = ["Enhancer", "Promoter", "UTR", "Coding", "Introns"]  # matches the post-filter set below
all_benchmarks = list(benchmark_colors.keys())

# --- Complete grid: one row per (category, benchmark), zero-filled where missing ---
full_grid = pl.DataFrame(
    [(c, b) for c in all_categories for b in all_benchmarks],
    schema=["consequence_category", "benchmark"],
    orient="row",
)

plot_df_full = (
    full_grid
    .join(
        plot_df.filter(~pl.col('consequence_category').is_in(['Intergenic', 'Splicing'])),
        on=["consequence_category", "benchmark"], how="left",
    )
    .with_columns(pl.col("num_variants").fill_null(0))
    # pseudocount so log10(0) doesn't blow up -- zero-count bars render as a thin
    # stub pinned at y=1, visually distinct from any real count (which starts at 2)
    .with_columns((pl.col("num_variants") + 1).alias("num_variants_plot"))
    .with_columns(
        pl.col("benchmark").cast(pl.Enum(all_benchmarks)),
        pl.col("consequence_category").cast(pl.Enum(all_categories)),
    )
)

# --- Plotting ---
plot = (
    ggplot(plot_df_full)
    + geom_bar(
        aes(x="consequence_category", y='num_variants_plot', fill='benchmark'),
        stat="identity",
        position=position_dodge2(preserve='single', padding=0.1),  # keeps bar width fixed even when a slot is missing/zero
        width=0.7,
        size=0.7,
    )
    + scale_fill_manual(values=benchmark_colors)
    + labs(x="Variant Region", y="Variants + 1")
    + scale_y_log10(labels=lambda breaks: [f'{int(round(b)):,}' for b in breaks])
    + theme_minimal()
    + theme(
        figure_size=(8, 4.33),
        axis_text=element_text(size=15),
        axis_title=element_text(size=15, lineheight=1.4),
        legend_text=element_text(size=13, lineheight=1.4),
        legend_title=element_text(size=0),
        legend_position='bottom',
        legend_direction='horizontal',
        legend_box='horizontal',
        plot_background=element_rect(fill="white", color="white"),
        panel_grid_major=element_line(color="#bfbfbf", size=0.6),
        panel_grid_minor=element_line(color="#d6d6d6", size=0.4),
    )
)

FIG_DIR = "PATH_TO_FILE"
plot.save(f"{FIG_DIR}/F1a_number_of_variants.svg", dpi=200)
plot